In [1]:
import os
os.environ["MODEL_KEY"]="ollama"
from dotenv import load_dotenv
load_dotenv()

from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, set_tracing_disabled

set_tracing_disabled(True)

In [2]:
local_model = OpenAIChatCompletionsModel(
    model="qwen3.5:cloud ",
    openai_client=AsyncOpenAI(
        api_key="ollama",
        base_url="http://localhost:11434/v1",
    )
)

In [3]:

agent = Agent(
    model=local_model,
    name="Greeter",                                    
    instructions="You are a friendly assistant. Be concise.",  
   )
result = await Runner.run(agent, "What is the capital of Pakistan?")
print(result.final_output)

The capital of Pakistan is Islamabad.


In [4]:
from agents import Agent, Runner

agent = Agent(name="Analyst", instructions="You analyze topics briefly.", model=local_model )

result = await Runner.run(agent, "Why is Python popular for AI?")

# The key properties of RunResult:
print("=" * 50)
print(f"Final Output:  {result.final_output[:100]}...")   # The text response
print(f"Last Agent:    {result.last_agent.name}")          # Which agent finished
print(f"New Items:     {len(result.new_items)}")           # Items generated during run
print(f"Raw Responses: {len(result.raw_responses)}")       # Raw LLM responses
print("=" * 50)

Final Output:  Python dominates AI development due to five key factors:

*   **Rich Ecosystem:** Libraries like Ten...
Last Agent:    Analyst
New Items:     1
Raw Responses: 1


In [5]:
agent = Agent(
    name="Tutor",
    instructions="You are a Python tutor. Be concise. Remember context from previous messages.",
    model=local_model
)

# Turn 1
result = await Runner.run(agent, "What is a list in Python?")
print(f"Turn 1: {result.final_output}\n")

# Turn 2 — carry conversation history forward
new_input = result.to_input_list() + [{"role": "user", "content": "How do I sort one?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 2: {result.final_output}\n")

# Turn 3
new_input = result.to_input_list() + [{"role": "user", "content": "What about reverse sort?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 3: {result.final_output}")

Turn 1: A **list** is an ordered, mutable collection of items.

```python
fruits = ["apple", "banana", "cherry"]
```

**Key Traits:**
*   **Ordered:** Items have indices (starting at 0).
*   **Mutable:** You can change, add, or remove items after creation.
*   **Flexible:** Can hold mixed data types (ints, strings, objects, etc.).

Access items using brackets: `fruits[0]` returns `"apple"`.

Turn 2: You have two main options:

**1. `.sort()` Method**
Modifies the original list directly.
```python
nums = [3, 1, 4]
nums.sort()
print(nums)  # [1, 3, 4]
```

**2. `sorted()` Function**
Returns a new sorted list, leaving the original unchanged.
```python
nums = [3, 1, 4]
new_nums = sorted(nums)
print(nums)      # [3, 1, 4]
print(new_nums)  # [1, 3, 4]
```

**Descending Order:**
Add `reverse=True` to either method.
```python
nums.sort(reverse=True)  # [4, 3, 1]
```

Turn 3: There are two ways to "reverse" a list, depending on your goal:

**1. Descending Sort (High to Low)**
Use `reverse=True`

In [6]:
from agents import Agent, Runner

agent = Agent(name="Streamer", instructions="Explain briefly.",model=local_model)

# Streaming — see tokens as they arrive
result = Runner.run_streamed(agent, "What is machine learning in 2 sentences?")

async for event in result.stream_events():
    if event.type == "raw_response_event" and hasattr(event.data, 'delta'):
        print(event.data.delta, end="", flush=True)

print(f"\n\n--- Final: {result.final_output}")

Thinking Process:

1.  **Analyze the Request:**
    *   Topic: Machine Learning (ML).
    *   Constraint: Explain briefly in exactly 2 sentences.

2.  **Identify Key Concepts of Machine Learning:**
    *   It's a subset of AI.
    *   It involves algorithms/models.
    *   It learns from data (rather than explicit programming).
    *   It improves performance over time/experience.
    *   It makes predictions or decisions.

3.  **Drafting - Attempt 1:**
    Machine learning is a branch of artificial intelligence where computers learn from data without being explicitly programmed for every task. By identifying patterns in large datasets, these systems can make predictions or decisions that improve over time with more experience.

4.  **Review - Attempt 1:**
    *   Sentence 1: "Machine learning is a branch of artificial intelligence where computers learn from data without being explicitly programmed for every task." (Good)
    *   Sentence 2: "By identifying patterns in large datasets, 